# Convert Scenario Data to MATLAB format (.mat)

This notebook processes PyTorch data files (`.pt`) for scenarios (scenario parameters, STFTs, features, and predictions) 
and converts them into standard `.mat` files which can be easily loaded in MATLAB via `load()`.

In [1]:
import torch
import numpy as np
import scipy.io as sio
from pathlib import Path
from tqdm import tqdm
import re

# Configuration Paths
debugstr = "" 
expname = f"Klaus_PALD_3D_1{debugstr}"
split = "test"

dataset_path = Path(f"/data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/{expname}")
pred_path = Path(f"/data4/Henri/j3/framewiseSpeakerCounting/predictions/{expname}_WGMSC_Feature_Extractor_GRU_estimator")
mat_output_path = Path(f"/data4/Henri/j3/framewiseSpeakerCounting/mat_files/{expname}")

# Create output directory
mat_output_path.mkdir(parents=True, exist_ok=True)

# Locate directories dynamically based on structure
stft_dirs = list((dataset_path / "stft").glob("*"))
feature_dirs = list((dataset_path / "features").glob("*"))

stft_dir = stft_dirs[0] if stft_dirs else None
feature_dir = feature_dirs[0] if feature_dirs else None



In [2]:
def make_valid_matlab_field(s):
    """Ensure dictionary keys are valid MATLAB struct field names."""
    s = str(s).replace("-", "_").replace(" ", "_")
    # If the string starts with a number, prefix it
    if re.match(r'^[0-9]', s):
        s = "str_" + s
    return s

def clean_for_matlab(obj):
    """Recursively clean Python objects for scipy.io.savemat."""
    if isinstance(obj, torch.Tensor):
        arr = obj.detach().cpu().numpy()
        return arr
    elif isinstance(obj, dict):
        return {make_valid_matlab_field(k): clean_for_matlab(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        if len(obj) > 0 and isinstance(obj[0], dict):
            # Reformat list of dicts to dict of lists for MATLAB struct array compatibility
            all_keys = set()
            for d in obj: 
                all_keys.update(d.keys())
            res = {make_valid_matlab_field(k): [] for k in all_keys}
            for d in obj:
                for k in all_keys:
                    res[make_valid_matlab_field(k)].append(clean_for_matlab(d.get(k, None)))
            # Transpose to appropriate lists if needed, but dict of lists handles it gracefully
            return res
        else:
            # Convert normal lists to numpy arrays if all elements are scalar, otherwise let savemat handle list of scalars/strings
            return [clean_for_matlab(v) for v in obj]
    elif isinstance(obj, Path):
        return str(obj)
    elif type(obj).__name__ == "Segment":
        return {
            "start": obj.start,
            "end": obj.end,
            "num_sources": obj.num_sources,
            "event_type": obj.event_type
        }
    elif type(obj).__name__ == "STFTtransform":
        return {
            "frame_length": getattr(obj, "frame_length", 0),
            "frame_shift": getattr(obj, "frame_shift", 0),
            "sampling_frequency": getattr(obj, "sampling_frequency", 0),
            "window_type": getattr(obj, "window_type", ""),
            "nfft": getattr(obj, "nfft", 0),
            "hop_length": getattr(obj, "hop_length", 0),
            "num_freq_bins": getattr(obj, "num_freq_bins", 0)
        }
    elif obj is None:
        return np.nan
    else:
        # Ints, floats, strings, etc.
        return obj

In [3]:
# Processing loop
success_count = 0
failed_scenarios = []

for split in ["test"]:
    scenario_files = list((dataset_path / split).glob("scenario_*.pt"))
    print(f"Found {len(scenario_files)} scenarios in split {split}.")
    for scenario_file in tqdm(scenario_files, desc="Converting to .mat"):
        scenario_id_str = scenario_file.stem.split('_')[1]
        
        out_file = mat_output_path / split / f"scenario_{scenario_id_str}.mat"
        
        if out_file.exists():
            print(f"Skipping scenario {scenario_id_str} (already exists).")
            continue
        
        # Paths to the other components
        stft_file = stft_dir / split / f"scenario_{scenario_id_str}.pt" if stft_dir else None
        feature_file = feature_dir / split / f"scenario_{scenario_id_str}.pt" if feature_dir else None
        
        pred_file_pattern = f"*_{split}_generator_{scenario_id_str}.pt"
        prediction_files = list(pred_path.glob(pred_file_pattern)) if pred_path.exists() else []
        prediction_file = prediction_files[0] if prediction_files else None
        
        # Load Pytorch dicts
        try:
            scenario_data = torch.load(scenario_file, weights_only=False) if scenario_file.exists() else {}
            stft_data = torch.load(stft_file, weights_only=False) if (stft_file and stft_file.exists()) else {}
            feature_data = torch.load(feature_file, weights_only=False) if (feature_file and feature_file.exists()) else {}
            pred_data = torch.load(prediction_file, weights_only=False) if (prediction_file and prediction_file.exists()) else {}
            
            # Assemble dictionary for scipy
            # Top-level variables in the mat file
            mat_dict = {
                "scenario_id": int(scenario_id_str)
            }
            
            if scenario_data:
                # We wrap it under standard keys
                mat_dict["scenario_data"] = clean_for_matlab(scenario_data)
            if stft_data:
                mat_dict["stft_data"] = clean_for_matlab(stft_data)
            if feature_data:
                mat_dict["feature_data"] = clean_for_matlab(feature_data)
            if pred_data is not None:
                mat_dict["prediction_data"] = clean_for_matlab(pred_data)
                
            
            
            # Save to MATLAB format
            sio.savemat(str(out_file), mat_dict)
            success_count += 1
            
        except Exception as e:
            failed_scenarios.append((scenario_id_str, str(e)))

    print(f"Successfully converted {success_count}/{len(scenario_files)} scenarios in split {split}.")
    if failed_scenarios:
        print(f"Failed to convert {len(failed_scenarios)} scenarios.")
        for sid, err in failed_scenarios[:5]:
            print(f"  - Scenario {sid}: {err}")
        if len(failed_scenarios) > 5:
            print("  ...")

Found 1500 scenarios in split train.


Converting to .mat:   0%|          | 0/1500 [02:26<?, ?it/s]


KeyboardInterrupt: 